# Cardiac MRI concept profiles — SAX and LAX

Restore the trained **ACDC central SAX** and **M&Ms-2 four-chamber LAX** stacked ED/ES ResNet50 models. Inspect the original images, extract final convolutional activations, identify cardiac concepts with Network Dissection, and visualize feature/concept importance for each image and target class.

Run from this repository with its Python dependencies and CineMA installed. The saved `architecture.json` supplies evaluation preprocessing; models use restored weights and evaluation mode throughout.

## 1. Paths and image selection

Defaults use the relocated sibling `Data and models` directory, with the SCAG results root as a fallback. Existing environment overrides remain available. `CLASS_NAME` selects the image's ground-truth class; `TARGET_CLASS=None` explains its predicted class, or set a class name such as `"HCM"`. Each view has its own class order and concept calibration.


In [ ]:
import json
import os
import sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf

PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "scag/config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the SCAG repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cinema.classification.dataset import get_image_transforms
from scag.activations import extract_activations
from scag.config import SCAGConfig
from scag.data import ACDCSemanticDataset, MNMS2SemanticDataset, _get_acdc_eval_transform, _load_acdc_splits, _load_mnms2_splits, _validate_splits, phase_encode_segments
from scag.models import EDES2DClassifier
from scag.semantics import compute_neuron_concept_matrix

SHARED_ROOT = PROJECT_ROOT.parent / "Data and models"
SAX_CFG = SCAGConfig(dataset="acdc")
LAX_CFG = SCAGConfig(dataset="mnms2")
DATA_BASE = SHARED_ROOT / "datasets" if (SHARED_ROOT / "datasets").is_dir() else SAX_CFG.results_root / "processed"
MODEL_BASE = SHARED_ROOT / "model checkpoints/2d" if (SHARED_ROOT / "model checkpoints/2d").is_dir() else SAX_CFG.results_root / "architecture_bank/2d"
SAX_DATA_ROOT = Path(os.environ.get("ACDC_PROCESSED_DIR", DATA_BASE / "acdc")).expanduser().resolve()
LAX_DATA_ROOT = Path(os.environ.get("MNMS2_PROCESSED_DIR", DATA_BASE / "mnms2")).expanduser().resolve()
SAX_CHECKPOINT = Path(os.environ.get("ACDC_CHECKPOINT", MODEL_BASE / "acdc_sax_mid_2d/resnet50/stacked/imagenet/seed_0/best_val_mcc.pt")).expanduser().resolve()
LAX_CHECKPOINT = Path(os.environ.get("MNMS2_LAX_CHECKPOINT", MODEL_BASE / "mnms2_lax_4c/resnet50/stacked/imagenet/seed_0/best_val_mcc.pt")).expanduser().resolve()
OUTPUT_ROOT = PROJECT_ROOT / "outputs/concept_profiles"
DEVICE = SAX_CFG.device
SPLIT = "train"  # Choose train, val, or test for displayed images.
SAX_CLASS_NAME = "HCM"  # None includes every class.
LAX_CLASS_NAME = "HCM"
SAX_N_SAMPLES = 4
LAX_N_SAMPLES = 4
SAX_TARGET_CLASS = None  # None explains each image's predicted class.
LAX_TARGET_CLASS = None
BATCH_SIZE = 4
ACTIVATION_TAIL = SAX_CFG.alpha  # Top 0.5% of calibration activations.
IOU_THRESHOLD = SAX_CFG.eta  # Shared SCAG detector cutoff: 0.015.
CONCEPT_NAMES = SAX_CFG.concept_names[1:]
PROFILE_NAMES = (*CONCEPT_NAMES, "Unassigned")

print("Device:", DEVICE)
print("SAX:", SAX_DATA_ROOT, SAX_CHECKPOINT)
print("LAX:", LAX_DATA_ROOT, LAX_CHECKPOINT)


## 2. Restore models and load images/masks

The same central SAX slice or LAX-4C plane is used for each image and its segmentation, with identical end padding. Segmentation labels are RV=1, myocardium=2, LV=3. Only evaluation transforms are applied, including to the calibration images from the training split.


In [ ]:
def load_view(cfg, data_root, checkpoint_path):
    architecture_path = checkpoint_path.parent / "architecture.json"
    for path in (checkpoint_path, architecture_path):
        if not path.is_file():
            raise FileNotFoundError(path)
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    architecture = json.loads(architecture_path.read_text())
    for key, expected in {"task": cfg.task_name, "classes": list(cfg.classes), "formulation": "stacked"}.items():
        if checkpoint.get(key) != expected or architecture.get(key) != expected:
            raise ValueError(f"Checkpoint/architecture {key} must match {expected!r}.")
    for key in ("backbone", "initialization"):
        if checkpoint[key] != architecture[key]:
            raise ValueError(f"Checkpoint and architecture disagree on {key}.")
    if not checkpoint["backbone"].startswith("resnet"):
        raise ValueError("Exact feature contributions require a stacked ResNet with a linear classifier.")
    model = EDES2DClassifier(checkpoint["backbone"], "stacked", "randinit", len(cfg.classes))
    model.load_state_dict(checkpoint["model"], strict=True)
    model.initialization = checkpoint["initialization"]
    model = model.to(DEVICE).eval()
    config = OmegaConf.create(architecture["config"])
    config.data.dir = str(data_root)
    if config.model.views != cfg.view or tuple(config.data[config.data.class_column]) != cfg.classes:
        raise ValueError("Saved preprocessing view/class order does not match the checkpoint.")
    adapter = SimpleNamespace(data_root=data_root, classes=cfg.classes, view=cfg.view)
    if cfg.dataset == "acdc":
        splits = _load_acdc_splits(adapter)
        transform = _get_acdc_eval_transform(config)
        datasets = {split: ACDCSemanticDataset(split, frame, transform, adapter) for split, frame in splits.items()}
    else:
        splits = _load_mnms2_splits(adapter)
        _, transform = get_image_transforms(config)
        datasets = {split: MNMS2SemanticDataset(split, frame, transform, adapter, config) for split, frame in splits.items()}
    _validate_splits(splits, cfg)
    print(f"{cfg.dataset}: restored {checkpoint['backbone']}; " + ", ".join(f"{split}={len(ds)}" for split, ds in datasets.items()))
    return model, model.encoder.layer4, datasets


def select_images(datasets, split, class_name, n_samples):
    if split not in datasets:
        raise ValueError(f"Unknown split: {split!r}")
    if type(n_samples) is not int or n_samples < 1:
        raise ValueError("n_samples must be a positive integer.")
    dataset = datasets[split]
    metadata = dataset.metadata
    if class_name is not None:
        class_name = class_name.strip().upper()
        if class_name not in set(metadata["pathology"]):
            raise ValueError(f"No {class_name!r} images in {split}.")
        metadata = metadata[metadata["pathology"] == class_name]
    indices = metadata.index[:n_samples].tolist()
    if not indices:
        raise ValueError("No images selected.")
    return [dataset[index] for index in indices]


sax_model, sax_layer, sax_datasets = load_view(SAX_CFG, SAX_DATA_ROOT, SAX_CHECKPOINT)
lax_model, lax_layer, lax_datasets = load_view(LAX_CFG, LAX_DATA_ROOT, LAX_CHECKPOINT)
sax_samples = select_images(sax_datasets, SPLIT, SAX_CLASS_NAME, SAX_N_SAMPLES)
lax_samples = select_images(lax_datasets, SPLIT, LAX_CLASS_NAME, LAX_N_SAMPLES)

for view, samples in (("SAX", sax_samples), ("LAX 4C", lax_samples)):
    fig, axes = plt.subplots(len(samples), 2, figsize=(8, 3 * len(samples)), squeeze=False)
    for row, (image, mask, info) in enumerate(samples):
        for phase_index, phase in enumerate(("ED", "ES")):
            axes[row, phase_index].imshow(image[phase_index], cmap="gray", vmin=0, vmax=1)
            axes[row, phase_index].set_title(f"{view} | {info['pid']} | {info['pathology']} | {phase}")
            axes[row, phase_index].axis("off")
    fig.tight_layout()
    plt.show()
    plt.close(fig)


## 3. Extract activations and dissect cardiac concepts

Reuse the repository's [Network Dissection](https://openaccess.thecvf.com/content_cvpr_2017/html/Bau_Network_Dissection_Quantifying_CVPR_2017_paper.html) procedure: estimate a per-channel upper-tail threshold over **all training-split images**, upsample each activation map, then measure dataset-wide intersection-over-union (IoU) against phase-encoded masks. The resulting `X` is a **channel × concept** matrix for RV, MYO and LV at ED and ES. `IOU_THRESHOLD` uses this project's cutoff; adjust it to change detector eligibility.

SAX and LAX are dissected separately. This fits semantic associations without updating model weights. The stacked ED/ES network produces joint spatial activations; phase labels describe mask associations, not separate phase-specific forward passes. Full-resolution dissection may take a few minutes on CPU.


In [ ]:
def dissect_view(model, layer, calibration_dataset):
    if not 0 < ACTIVATION_TAIL < 1 or not 0 < IOU_THRESHOLD <= 1:
        raise ValueError("Require 0 < ACTIVATION_TAIL < 1 and 0 < IOU_THRESHOLD <= 1.")
    loader = torch.utils.data.DataLoader(calibration_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    activations, masks = extract_activations(model, loader, layer, DEVICE)
    if masks is None or masks.shape[1] != 2 or not torch.all((masks >= 0) & (masks <= 3)):
        raise ValueError("Dissection requires aligned ED/ES masks with labels 0, 1, 2, 3.")
    X, labeled = compute_neuron_concept_matrix(activations, phase_encode_segments(masks), 7, ACTIVATION_TAIL, IOU_THRESHOLD)
    # A channel with no response cannot be a semantic detector, even when its threshold is zero.
    active = activations.amax(dim=(0, 2, 3)).numpy() > 0
    X[~active] = 0
    labeled &= active
    print(f"Activations: {tuple(activations.shape)}; labeled channels: {labeled.sum()}/{len(labeled)}")
    return X, labeled


sax_X, sax_labeled = dissect_view(sax_model, sax_layer, sax_datasets["train"])
lax_X, lax_labeled = dissect_view(lax_model, lax_layer, lax_datasets["train"])

detector_counts = pd.DataFrame({"concept": CONCEPT_NAMES, "SAX detectors": (sax_X[:, 1:] >= IOU_THRESHOLD).sum(axis=0), "LAX detectors": (lax_X[:, 1:] >= IOU_THRESHOLD).sum(axis=0)})
print(detector_counts.to_string(index=False))


## 4. Image/class concept profiles

For final-layer channel `k` and class `c`, **feature contribution = spatial mean activation[k] × classifier weight[c,k]**. For these stacked ResNets, feature contributions plus the classifier bias exactly reconstruct the class logit; no gradients are needed.

For semantic display, normalize eligible IoUs within each labeled channel and distribute its contribution among the six concepts. IoUs below the detector cutoff are excluded. Unlabeled channels go into **Unassigned**, preserving the full score. This semantic allocation is an interpretation of channel associations, rather than a causal effect of an anatomical region. Positive values support the target logit and negative values oppose it; values are logit units, not probabilities. The classifier bias is reported separately.


In [ ]:
def concept_allocation(X, labeled, iou_threshold=IOU_THRESHOLD):
    X = np.asarray(X, dtype=np.float32)
    labeled = np.asarray(labeled, dtype=bool)
    if X.ndim != 2 or X.shape[1] != 7 or labeled.shape != (len(X),):
        raise ValueError("Expected channel × 7 IoUs and one detector flag per channel.")
    if not np.isfinite(X).all() or np.any((X < 0) | (X > 1)):
        raise ValueError("IoUs must be finite and between zero and one.")
    association = np.where(labeled[:, None] & (X[:, 1:] >= iou_threshold), X[:, 1:], 0)
    totals = association.sum(axis=1, keepdims=True)
    association = np.divide(association, totals, out=np.zeros_like(association), where=totals > 0)
    return np.column_stack((association, (totals[:, 0] == 0).astype(np.float32)))


def image_profiles(model, layer, samples, X, labeled, classes, target_class=None):
    if target_class is not None and target_class.strip().upper() not in classes:
        raise ValueError(f"Target class must be one of {classes}.")
    loader = torch.utils.data.DataLoader(samples, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    activations, _ = extract_activations(model, loader, layer, DEVICE)
    pooled = activations.mean(dim=(2, 3)).numpy()
    weight = model.classifier.weight.detach().cpu().numpy()
    bias = model.classifier.bias.detach().cpu().numpy()
    feature_scores = pooled[:, None, :] * weight[None, :, :]
    logits = feature_scores.sum(axis=2) + bias
    with torch.no_grad():
        actual_logits = torch.cat([model(batch[0].to(DEVICE)).cpu() for batch in loader]).numpy()
    np.testing.assert_allclose(logits, actual_logits, rtol=1e-4, atol=1e-4)
    allocation = concept_allocation(X, labeled)
    profiles = feature_scores @ allocation
    np.testing.assert_allclose(profiles.sum(axis=2) + bias, logits, rtol=1e-4, atol=1e-4)
    predictions = logits.argmax(axis=1)
    targets = predictions if target_class is None else np.full(len(samples), classes.index(target_class.strip().upper()))
    probabilities = torch.from_numpy(logits).softmax(dim=1).numpy()
    rows = []
    for index, (_, _, info) in enumerate(samples):
        target = int(targets[index])
        rows.append({
            "pid": info["pid"], "image_class": info["pathology"], "predicted_class": classes[predictions[index]],
            "target_class": classes[target], "target_probability": float(probabilities[index, target]),
            "target_logit": float(logits[index, target]), "classifier_bias": float(bias[target]),
            **dict(zip(PROFILE_NAMES, profiles[index, target].tolist())),
        })
    return {"activations": activations, "feature_scores": feature_scores, "profiles": profiles, "targets": targets, "table": pd.DataFrame(rows)}


# Runnable arithmetic check: mixed concepts, negative evidence and unassigned channels conserve the score.
check_X = np.zeros((3, 7), dtype=np.float32)
check_X[0, 1], check_X[0, 2], check_X[1, 3] = 0.1, 0.3, 0.2
check_allocation = concept_allocation(check_X, np.array([True, True, False]), iou_threshold=0.015)
np.testing.assert_allclose(check_allocation.sum(axis=1), 1)
check_scores = np.array([4., -2., 3.]) @ check_allocation
np.testing.assert_allclose(check_scores, [1, 3, -2, 0, 0, 0, 3])
assert np.isclose(check_scores.sum(), 5)
assert np.all(concept_allocation(check_X, np.zeros(3, dtype=bool))[:, -1] == 1)
print("Concept allocation self-check passed.")


## 5. Visualize images, concept importance and feature importance

Each row shows ED/ES originals, the complete signed concept profile, and the ten features with the largest absolute contributions. Feature labels use their strongest eligible concept association, or Unassigned. View-specific scores use the same axis scale across displayed patients.


In [ ]:
def plot_profiles(samples, result, X, labeled, view):
    table = result["table"]
    concept_limit = max(float(np.abs(table[list(PROFILE_NAMES)].to_numpy()).max()), 0.01) * 1.1
    selected_features = result["feature_scores"][np.arange(len(samples)), result["targets"]]
    feature_limit = max(float(np.abs(selected_features).max()), 0.01) * 1.1
    fig, axes = plt.subplots(len(samples), 4, figsize=(21, 4 * len(samples)), squeeze=False, gridspec_kw={"width_ratios": [1, 1, 1.5, 1.7]})
    for row, (image, _, info) in enumerate(samples):
        record = table.iloc[row]
        for phase_index, phase in enumerate(("ED", "ES")):
            axes[row, phase_index].imshow(image[phase_index], cmap="gray", vmin=0, vmax=1)
            axes[row, phase_index].set_title(f"{view} | {info['pid']} | {info['pathology']} | {phase}")
            axes[row, phase_index].axis("off")
        scores = record[list(PROFILE_NAMES)].to_numpy(dtype=float)
        ax = axes[row, 2]
        ax.barh(PROFILE_NAMES, scores, color=np.where(scores >= 0, "#218c74", "#c0392b"))
        ax.invert_yaxis()
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_xlim(-concept_limit, concept_limit)
        ax.set_xlabel("Contribution to target logit")
        ax.set_title(f"Target {record['target_class']} ({record['target_probability']:.1%})\nPrediction {record['predicted_class']} | bias {record['classifier_bias']:.3f}")
        features = selected_features[row]
        top = np.argsort(np.abs(features))[-10:][::-1]
        names = [f"#{k}: {CONCEPT_NAMES[X[k, 1:].argmax()] if labeled[k] else 'Unassigned'}" for k in top]
        ax = axes[row, 3]
        ax.barh(names, features[top], color=np.where(features[top] >= 0, "#218c74", "#c0392b"))
        ax.invert_yaxis()
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_xlim(-feature_limit, feature_limit)
        ax.set_xlabel("Contribution to target logit")
        ax.set_title("Top 10 features by absolute contribution")
    fig.tight_layout()
    return fig


sax_result = image_profiles(sax_model, sax_layer, sax_samples, sax_X, sax_labeled, SAX_CFG.classes, SAX_TARGET_CLASS)
lax_result = image_profiles(lax_model, lax_layer, lax_samples, lax_X, lax_labeled, LAX_CFG.classes, LAX_TARGET_CLASS)
for view, samples, result, X, labeled in (("SAX", sax_samples, sax_result, sax_X, sax_labeled), ("LAX 4C", lax_samples, lax_result, lax_X, lax_labeled)):
    print(view + " image/class profiles:")
    print(result["table"].round(4).to_string(index=False))
    fig = plot_profiles(samples, result, X, labeled, view)
    plt.show()
    plt.close(fig)


## 6. See what each concept detector activates on

For the first selected image in each view, display the channel with the highest eligible IoU for each concept. These are **individual forward activation maps**, with the corresponding ground-truth anatomy outlined in cyan. Each map has its own intensity scale. An absent eligible detector is shown explicitly. These joint ED/ES maps visualize association; they are not class contribution maps.


In [ ]:
def plot_concept_activations(sample, activation, X, labeled, view):
    image, mask, info = sample
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for concept_index, (name, ax) in enumerate(zip(CONCEPT_NAMES, axes.flat), start=1):
        phase_index = (concept_index - 1) // 3
        anatomy_label = (concept_index - 1) % 3 + 1
        ax.imshow(image[phase_index], cmap="gray", vmin=0, vmax=1)
        candidates = np.flatnonzero(labeled & (X[:, concept_index] >= IOU_THRESHOLD))
        if len(candidates):
            channel = int(candidates[np.argmax(X[candidates, concept_index])])
            heatmap = F.interpolate(activation[channel][None, None], size=image.shape[-2:], mode="bilinear", align_corners=False)[0, 0].numpy()
            overlay = ax.imshow(heatmap, cmap="magma", alpha=0.5)
            fig.colorbar(overlay, ax=ax, fraction=0.046, pad=0.04)
            ax.set_title(f"{name} | feature #{channel} | IoU {X[channel, concept_index]:.3f}")
        else:
            ax.set_title(f"{name} | no eligible detector")
        region = mask[phase_index].numpy() == anatomy_label
        if region.any() and not region.all():
            ax.contour(region, levels=[0.5], colors=["cyan"], linewidths=0.8)
        ax.axis("off")
    fig.suptitle(f"{view} | {info['pid']} | {info['pathology']} — concept detector activations")
    fig.tight_layout()
    return fig


for view, samples, result, X, labeled in (("SAX", sax_samples, sax_result, sax_X, sax_labeled), ("LAX 4C", lax_samples, lax_result, lax_X, lax_labeled)):
    fig = plot_concept_activations(samples[0], result["activations"][0], X, labeled, view)
    plt.show()
    plt.close(fig)


## 7. Save originals, activation arrays and profiles

Each view gets its own folder under `outputs/concept_profiles/`, with ED/ES original PNGs, profile and detector figures, a CSV for the selected target classes, and an NPZ containing selected-image activations, IoUs, all-class feature/concept scores, patient IDs, and class/concept order. `mean_profiles_by_image_class.csv` averages the displayed cohort for every possible target class; it describes only these selected images. SAX and LAX remain separate because they use different models and pathology vocabularies.


In [ ]:
def export_view(view, samples, result, X, labeled, classes, checkpoint_path):
    output_dir = OUTPUT_ROOT / view / SPLIT
    output_dir.mkdir(parents=True, exist_ok=True)
    for image, _, info in samples:
        for phase_index, phase in enumerate(("ED", "ES")):
            plt.imsave(output_dir / f"{info['pid']}_{phase}_original.png", image[phase_index].numpy(), cmap="gray", vmin=0, vmax=1)
    fig = plot_profiles(samples, result, X, labeled, view)
    fig.savefig(output_dir / "image_concept_profiles.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    fig = plot_concept_activations(samples[0], result["activations"][0], X, labeled, view)
    fig.savefig(output_dir / "concept_detector_activations.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    result["table"].to_csv(output_dir / "image_concept_profiles.csv", index=False)
    image_classes = np.array([info["pathology"] for _, _, info in samples])
    summary = []
    for image_class in sorted(set(image_classes)):
        members = image_classes == image_class
        for class_index, target_class in enumerate(classes):
            scores = result["profiles"][members, class_index].mean(axis=0)
            summary.append({"image_class": image_class, "target_class": target_class, "n_images": int(members.sum()), **dict(zip(PROFILE_NAMES, scores.tolist()))})
    pd.DataFrame(summary).to_csv(output_dir / "mean_profiles_by_image_class.csv", index=False)
    np.savez_compressed(
        output_dir / "activations_and_concept_profiles.npz",
        activations=result["activations"].numpy(), X=X, labeled=labeled, allocation=concept_allocation(X, labeled),
        feature_scores=result["feature_scores"], concept_profiles=result["profiles"], target_indices=result["targets"],
        patient_ids=np.array([info["pid"] for _, _, info in samples]), image_classes=image_classes,
        classes=np.array(classes), concept_names=np.array(("background", *CONCEPT_NAMES)), profile_names=np.array(PROFILE_NAMES),
        classifier_bias=result["table"]["classifier_bias"].to_numpy(), checkpoint=str(checkpoint_path),
        feature_layer="encoder.layer4", calibration_split="train", calibration_class_filter="all", activation_tail=ACTIVATION_TAIL, iou_threshold=IOU_THRESHOLD,
    )
    print("Saved", output_dir)


export_view("acdc_sax", sax_samples, sax_result, sax_X, sax_labeled, SAX_CFG.classes, SAX_CHECKPOINT)
export_view("mnms2_lax_4c", lax_samples, lax_result, lax_X, lax_labeled, LAX_CFG.classes, LAX_CHECKPOINT)
